# 02 · DINOv3 feature extraction (Phase 4)

Đây là notebook độc lập cho một Kaggle kernel mới. Notebook tự cài dependency, clone repo, tìm dataset, xác thực Hugging Face, tái tạo split Phase 2 và load checkpoint; không dùng biến, model hoặc cache trong kernel của notebook 01.

## Trước khi chạy

1. Chọn GPU trong Kaggle Notebook Settings.
2. Add Input bộ Who Is AI có `train/manifest.csv` và `train/images/`.
3. Đảm bảo repo trên GitHub đã có code mới nhất (`src/`, `scripts/create_splits.py`).
4. Trên Hugging Face, tự chấp nhận điều khoản và xin quyền checkpoint DINOv3; thêm Kaggle Secret tên `HF_TOKEN`.
5. Bật Internet để cài package và tải checkpoint nếu chưa được cache.

Cache được lưu vào `/kaggle/working/phase4_features/dinov3_vitb16_seed42/`, gồm global CLS feature, patch tokens, labels, metadata, processor config và split manifests. Feature được lưu `float16` để giảm dung lượng; model ID, preprocessing và dtype được ghi trong `cache_manifest.json`.

In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/FaceTrace.git'
REPO_REF = 'main'
REPO_DIR = Path('/kaggle/working/FaceTrace')

# Prefer an attached repository input; otherwise clone a clean working copy.
attached_candidates = [Path('/kaggle/input/facetrace'), Path('/kaggle/input/face-trace')]
attached_repo = next((p for p in attached_candidates if (p / 'src').is_dir()), None)
if attached_repo is not None:
    REPO_DIR = attached_repo
elif not (REPO_DIR / 'src').is_dir():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        REPO_DIR = Path('/kaggle/working/FaceTrace_repo')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

required_paths = [REPO_DIR / 'src/models/dinov3.py', REPO_DIR / 'src/input_data/manifest_dataset.py', REPO_DIR / 'scripts/create_splits.py']
missing = [str(p) for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError(f'Repo snapshot is missing required files: {missing}')
sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR)

Cloning into '/kaggle/working/FaceTrace'...


Repo: /kaggle/working/FaceTrace


In [2]:
%pip install -q 'transformers>=4.56.1,<5' 'PyYAML>=6' 'pandas>=2' 'Pillow>=10' 'tqdm>=4.66' 'scikit-learn>=1.4'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import random
import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError('GPU chưa được bật. Chọn Settings → Accelerator → GPU rồi chạy lại.')

SEED = 42
MODEL_ID = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = torch.device('cuda')
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Model:', MODEL_ID)

GPU: Tesla T4
PyTorch: 2.10.0+cu128
Transformers: 4.57.6
Model: facebook/dinov3-vitb16-pretrain-lvd1689m


In [4]:
from pathlib import Path

# Set DATA_ROOT manually if the mounted dataset uses a different layout.
DATA_ROOT = None
if DATA_ROOT is None:
    input_root = Path('/kaggle/input')
    manifests = sorted(input_root.rglob('manifest.csv')) if input_root.exists() else []
    matches = [p for p in manifests if p.parent.name == 'train' and (p.parent / 'images').is_dir()]
    if not matches:
        raise FileNotFoundError('Không tìm thấy train/manifest.csv cùng train/images/. Add Input Who Is AI hoặc đặt DATA_ROOT thủ công.')
    DATA_ROOT = matches[0].parent.parent
else:
    DATA_ROOT = Path(DATA_ROOT)

TRAIN_ROOT = DATA_ROOT / 'train'
MANIFEST_PATH = TRAIN_ROOT / 'manifest.csv'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(MANIFEST_PATH)
print('Dataset:', DATA_ROOT)
print('Train manifest:', MANIFEST_PATH)

Dataset: /kaggle/input/datasets/dinhbaobao/who-is-ai/data
Train manifest: /kaggle/input/datasets/dinhbaobao/who-is-ai/data/train/manifest.csv


In [5]:
import os

HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = HF_TOKEN or UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

if not HF_TOKEN:
    raise RuntimeError('Thêm Kaggle Secret tên HF_TOKEN sau khi tài khoản Hugging Face được cấp quyền model.')

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
del HF_TOKEN
print('Hugging Face authentication configured; token hidden.')

Hugging Face authentication configured; token hidden.


In [6]:
import importlib.util
import pandas as pd

split_spec = importlib.util.spec_from_file_location('phase2_split_generator', REPO_DIR / 'scripts' / 'create_splits.py')
split_module = importlib.util.module_from_spec(split_spec)
split_spec.loader.exec_module(split_module)

manifest_rows = split_module.read_manifest(MANIFEST_PATH)
train_rows, val_rows = split_module.create_splits(
    manifest_rows,
    validation_fraction=0.20,
    seed=SEED,
    train_images_root=TRAIN_ROOT,
    image_path_base=REPO_DIR,
)

SPLIT_DIR = Path('/kaggle/working/phase4_splits')
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = SPLIT_DIR / 'train.csv'
VAL_CSV = SPLIT_DIR / 'val.csv'
split_module.write_csv(TRAIN_CSV, train_rows)
split_module.write_csv(VAL_CSV, val_rows)

counts = pd.DataFrame(train_rows + val_rows).groupby(['split', 'label', 'class_name'], dropna=False).size().rename('n_images').reset_index()
display(counts)
assert len(train_rows) == 1600 and len(val_rows) == 400
assert len({r['image_id'] for r in train_rows}.intersection(r['image_id'] for r in val_rows)) == 0

,split,label,class_name,n_images
0,train,0,Real,800
1,train,1,Fake,800
2,val,0,Real,200
3,val,1,Fake,200


In [7]:
from transformers import AutoImageProcessor
from src.models.dinov3 import DINOv3FeatureExtractor

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
processor.size = {'height': IMAGE_SIZE, 'width': IMAGE_SIZE}
extractor = DINOv3FeatureExtractor(MODEL_ID, freeze=True)
extractor.to(DEVICE)
extractor.eval()

print('Hidden size:', extractor.hidden_size)
print('Register tokens:', getattr(extractor.backbone.config, 'num_register_tokens', 0))
print('Resolved model commit:', getattr(extractor.backbone.config, '_commit_hash', None))

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Hidden size: 768
Register tokens: 4
Resolved model commit: 5931719e67bbdb9737e363e781fb0c67687896bc


In [8]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from src.input_data.manifest_dataset import ManifestImageDataset

def extract_manifest(manifest_path: Path, split_name: str) -> dict:
    dataset = ManifestImageDataset(
        manifest_path,
        processor=processor,
        image_column='image_path',
        label_column='label',
        image_path_base=REPO_DIR,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    global_parts, patch_parts, label_parts = [], [], []
    metadata = []
    with torch.inference_mode():
        for batch in tqdm(loader, desc=f'extract {split_name}'):
            pixels = batch['pixel_values'].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                output = extractor(pixels)
            global_parts.append(output['global'].cpu().to(torch.float16))
            patch_parts.append(output['patches'].cpu().to(torch.float16))
            label_parts.append(batch['label'].cpu())
            metadata.extend(
                {
                    'image_id': image_id,
                    'image_path': image_path,
                    'label': int(label),
                    'source': source,
                    'split': split_name,
                }
                for image_id, image_path, label, source in zip(
                    batch['image_id'], batch['image_path'], batch['label'].tolist(), batch['source']
                )
            )
    return {
        'global': torch.cat(global_parts),
        'patches': torch.cat(patch_parts),
        'labels': torch.cat(label_parts),
        'metadata': metadata,
    }

train_features = extract_manifest(TRAIN_CSV, 'train')
val_features = extract_manifest(VAL_CSV, 'val')
global_features = torch.cat([train_features['global'], val_features['global']])
patch_features = torch.cat([train_features['patches'], val_features['patches']])
labels = torch.cat([train_features['labels'], val_features['labels']])
metadata = pd.DataFrame(train_features['metadata'] + val_features['metadata'])
print('Global shape:', tuple(global_features.shape))
print('Patch shape:', tuple(patch_features.shape))
print('Labels:', tuple(labels.shape))

extract train:   0%|          | 0/50 [00:00<?, ?it/s]

extract val:   0%|          | 0/13 [00:00<?, ?it/s]

Global shape: (2000, 768)
Patch shape: (2000, 196, 768)
Labels: (2000,)


In [9]:
import json
import shutil

CACHE_DIR = Path('/kaggle/working/phase4_features/dinov3_vitb16_seed42')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert len(metadata) == len(global_features) == len(patch_features) == len(labels)
assert metadata['image_id'].is_unique
assert metadata.groupby('split').size().to_dict() == {'train': 1600, 'val': 400}

torch.save(global_features, CACHE_DIR / 'global_features.pt')
torch.save(patch_features, CACHE_DIR / 'patch_features.pt')
torch.save(labels, CACHE_DIR / 'labels.pt')
metadata.to_csv(CACHE_DIR / 'metadata.csv', index=False)
shutil.copy2(TRAIN_CSV, CACHE_DIR / 'train.csv')
shutil.copy2(VAL_CSV, CACHE_DIR / 'val.csv')
(CACHE_DIR / 'processor.json').write_text(json.dumps(processor.to_dict(), indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
cache_manifest = {
    'model_id': MODEL_ID,
    'model_revision': getattr(extractor.backbone.config, '_commit_hash', None),
    'seed': SEED,
    'image_size': IMAGE_SIZE,
    'feature_dtype': 'float16',
    'global_shape': list(global_features.shape),
    'patch_shape': list(patch_features.shape),
    'labels_shape': list(labels.shape),
    'patch_tokens_exclude_cls_and_registers': True,
    'split_counts': metadata['split'].value_counts().to_dict(),
    'source_metadata_available': False,
    'evaluation_claim': 'in-domain/protocol sanity check only',
}
(CACHE_DIR / 'cache_manifest.json').write_text(json.dumps(cache_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

archive = shutil.make_archive('/kaggle/working/phase4_dinov3_feature_cache', 'zip', root_dir=CACHE_DIR)
print('Cache directory:', CACHE_DIR)
print('Artifact ZIP:', archive)
print(cache_manifest)

Cache directory: /kaggle/working/phase4_features/dinov3_vitb16_seed42
Artifact ZIP: /kaggle/working/phase4_dinov3_feature_cache.zip
{'model_id': 'facebook/dinov3-vitb16-pretrain-lvd1689m', 'model_revision': '5931719e67bbdb9737e363e781fb0c67687896bc', 'seed': 42, 'image_size': 224, 'feature_dtype': 'float16', 'global_shape': [2000, 768], 'patch_shape': [2000, 196, 768], 'labels_shape': [2000], 'patch_tokens_exclude_cls_and_registers': True, 'split_counts': {'train': 1600, 'val': 400}, 'source_metadata_available': False, 'evaluation_claim': 'in-domain/protocol sanity check only'}
